# Part 6: Reporting & Dashboards

**vfairness.operations.reporting** — Interactive dashboards, automated reports

The vfairness library is organized into eight parts following the ML fairness pipeline:
- **Part 1 — Data & Preprocessing** (`vfairness.preprocessing`) — Feature transforms, bias auditing
- **Part 2 — Training-Time Interventions** (`vfairness.in_processing`) — Losses, constraints, regularizers
- **Part 3 — Prediction-Time Interventions** (`vfairness.post_processing`) — Calibration, threshold tuning
- **Part 4 — Evaluation & Measurement** (`vfairness.evaluation`) — Metrics, statistics, visualization
- **Part 5 — Operations & Monitoring** (`vfairness.operations`) — MLOps, real-time monitoring, drift detection
- **Part 6 — Reporting & Dashboards** (`vfairness.operations.reporting`) — Interactive dashboards, automated reports
- **Part 7 — Experimentation** (`vfairness.operations.experimentation`) — A/B testing, Pareto & causal
- **Part 8 — Workflow Integration** (`vfairness.operations.cicd`) — CI/CD gates, pre-commit hooks, pytest plugin, report cards

> This notebook covers **Part 6** — performance dashboards and reporting layer.

---

This notebook demonstrates the reporting components of `vfairness.operations.reporting`:

| Component | Class | Purpose |
|-----------|-------|---------|
| Data Layer | `MetricsStore` | Unified ingestion + privacy-preserving query API |
| Visualization | `FairnessDashboard` | Plotly-based progressive-disclosure dashboards |
| Reporting | `ReportGenerator` | Multi-tier, multi-format automated reports with NLG |
| Interactivity | `InteractiveDashboard` | Dash app / standalone HTML with what-if analysis |

This notebook builds on the monitoring infrastructure from [Part 5](vfairness_5_monitoring_demo.ipynb).
We create synthetic monitoring data to demonstrate the reporting layer.

## 0 · Setup

In [ ]:
import sys, pathlib
# Ensure the vfairness source tree is on the path
repo_root = pathlib.Path.cwd().parent
src_path  = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

rng = np.random.default_rng(seed=42)

# Check optional dependencies
try:
    import plotly
    PLOTLY_AVAILABLE = True
    print(f'Plotly {plotly.__version__} available')
except ImportError:
    PLOTLY_AVAILABLE = False
    print('Plotly not available — dashboard/chart demos will be skipped')

print('Setup complete.')

In [ ]:
# Import all reporting components
from vfairness.operations.reporting import (
    MetricsStore, MetricsStoreConfig, HealthScore, PrivacyLevel,
    FairnessDashboard, DashboardConfig, TimeWindow,
    ReportGenerator, ReportConfig, ReportTier, OutputFormat, GeneratedReport,
    InteractiveDashboard, InteractiveConfig, simulate_threshold_change,
)

# Import monitoring components for ingestion demos
from vfairness.operations.monitoring import (
    FairnessMonitor, FairnessMonitorConfig, WindowMetrics,
    TemporalFairnessAnalyzer,
    FairnessDriftDetector,
    AdaptiveThresholdManager,
    FairnessAlertPrioritizer,
)

print('All reporting imports successful.')

## 1 · Synthetic Monitoring Data

We generate 30 days of synthetic fairness monitoring data using the same components
from Parts 1 & 2. This feeds the MetricsStore, which in turn powers all dashboards
and reports.

In [ ]:
# ------------------------------------------------------------------
# Generate 30 days of synthetic batch data
# ------------------------------------------------------------------
n_days = 30
n_batches_per_day = 4
n_samples = 500
groups = ['male', 'female', 'non_binary']

# `prediction_col` and `label_col` MUST name the columns this frame actually
# uses. They default to 'prediction' and 'label', and this demo's frame uses
# 'y_pred' and 'y_true'. Until 2026-09-10 they were left at the defaults, so
# every metric computed to NaN, no threshold comparison was ever recorded, and
# the whole reporting stack below was demonstrated over a monitor that had
# measured nothing across all 120 windows. The monitor now warns when the
# configured prediction column is absent, which is how this was found.
monitor = FairnessMonitor(
    config=FairnessMonitorConfig(
        window_size=500,
        alert_threshold=0.80,
        prediction_col='y_pred',
        label_col='y_true',
    )
)

for day in range(n_days):
    for batch in range(n_batches_per_day):
        # Simulate group predictions with slight drift over time
        drift_factor = 0.02 * day / n_days  # gradual drift
        data = {
            'y_true': rng.binomial(1, 0.5, n_samples),
            'y_pred': rng.binomial(1, 0.5 + drift_factor, n_samples),
            'group_gender': rng.choice(groups, n_samples, p=[0.45, 0.45, 0.10]),
        }
        df_batch = pd.DataFrame(data)
        monitor.update_and_check(df_batch)

print(f'Generated {n_days * n_batches_per_day} batches over {n_days} days')
print(f'Monitor history: {len(monitor.get_metric_history())} records')

In [ ]:
# ------------------------------------------------------------------
# Create Temporal Analyzer with daily data
# ------------------------------------------------------------------
analyzer = TemporalFairnessAnalyzer()

for day in range(n_days):
    date = pd.Timestamp.now().normalize() - pd.Timedelta(days=n_days - day)
    drift_factor = 0.02 * day / n_days
    analyzer.update_daily_metrics(date, {
        'demographic_parity': max(0.0, 0.85 - drift_factor + rng.normal(0, 0.03)),
        'equalized_odds': max(0.0, 0.90 - drift_factor * 0.5 + rng.normal(0, 0.02)),
        'predictive_parity': max(0.0, 0.88 + rng.normal(0, 0.02)),
    })

print(f'Analyzer daily records: {len(analyzer.to_dataframe())}')

In [ ]:
# ------------------------------------------------------------------
# Create Drift Detector with baseline + current data
# ------------------------------------------------------------------
detector = FairnessDriftDetector()

# Baseline: first 15 days
baseline_series = pd.Series([0.85 + rng.normal(0, 0.02) for _ in range(15)])
detector.set_baseline(baseline_series)

# Current: last 15 days with drift
current_series = pd.Series([0.82 + rng.normal(0, 0.03) for _ in range(15)])
drift_result = detector.check_drift(current_series, metric='demographic_parity')
print(f'Drift detected: {drift_result.drift_detected}')
print(f'Overall drift score: {drift_result.overall_drift_score:.4f}')

In [ ]:
# ------------------------------------------------------------------
# Create Alerts via Prioritizer
# ------------------------------------------------------------------
prioritizer = FairnessAlertPrioritizer()

# Simulate a few alert events
events = [
    FairnessAlertPrioritizer.build_drift_event(
        metric_name='demographic_parity',
        affected_groups=['female', 'non_binary'],
        drift_score=0.65,
        mean_shift=0.04,
        regulatory_risk=0.8,
        historical_discrimination=0.7,
    ),
    FairnessAlertPrioritizer.build_drift_event(
        metric_name='equalized_odds',
        affected_groups=['non_binary'],
        drift_score=0.35,
        mean_shift=0.02,
        regulatory_risk=0.5,
        historical_discrimination=0.3,
    ),
]

alerts = [prioritizer.create_alert(e) for e in events]
for a in alerts:
    print(f'Alert: {a.severity} — {a.metric_name} (priority: {a.priority_score:.1f})')

## 2 · MetricsStore — Unified Data Layer

The `MetricsStore` is the single source of truth that powers every dashboard,
report, and interactive view. It ingests data from all five monitoring components
and provides a consistent query API with built-in privacy protection.

### Three-tier privacy scheme

| Group Size | Protection | Mechanism |
|------------|------------|-----------|
| < k (10) | **Suppressed** | Values replaced with NaN |
| k – 50 | **Noisy** | Laplace noise (ε-differential privacy) |
| > 50 | **Exact** | No modification |

In [ ]:
# ------------------------------------------------------------------
# Create MetricsStore and ingest from all components
# ------------------------------------------------------------------
store = MetricsStore(config=MetricsStoreConfig(
    k_anonymity_threshold=10,
    noisy_threshold=50,
    dp_epsilon=1.0,
    enable_privacy=True,
))

# Ingest from all monitoring components
n_monitor = store.ingest_from_monitor(monitor)
n_analyzer = store.ingest_from_analyzer(analyzer)
n_drift = store.ingest_drift_result(drift_result)
n_alerts = sum(store.ingest_alert(a) for a in alerts)

print(f'Ingested from FairnessMonitor:         {n_monitor:>5} records')
print(f'Ingested from TemporalFairnessAnalyzer: {n_analyzer:>5} records')
print(f'Ingested from FairnessDriftDetector:    {n_drift:>5} records')
print(f'Ingested from FairnessAlertPrioritizer: {n_alerts:>5} records')
print(f'\nTotal store size: {len(store)} records')
print(f'\nStore summary:')
for k, v in store.get_summary().items():
    print(f'  {k}: {v}')

### Querying the Store

The `get_metrics()` method returns a tidy DataFrame with privacy applied by default.
Filters for time range, metric names, groups, and sources are all optional.

In [ ]:
# Query with privacy applied (default)
df = store.get_metrics(
    metrics=['demographic_parity'],
    start_time=datetime.now() - timedelta(days=7),
)
print(f'Query returned {len(df)} records')
print(f'Privacy levels: {df["privacy_level"].value_counts().to_dict()}')
df.head(10)

### Health Score

The composite **Fairness Health Score** (0–100) aggregates three components:
- **Metric compliance** (50%): fraction of recent metrics without alerts
- **Alert frequency** (30%): penalty for each alert in the window
- **Drift stability** (20%): fraction of windows without drift

The score maps to a traffic-light status: green (≥80), yellow (50–79), red (<50).

In [ ]:
# Compute health score
hs = store.compute_health_score(time_window=timedelta(days=7))

print(f'Health Score: {hs.score:.1f}/100')
print(f'Status:       {hs.status}')
print(f'Trend:        {hs.trend} (slope: {hs.trend_slope:+.4f}/day)')
print(f'Components:   {hs.components}')
print(f'Alerts:       {hs.n_alerts}')
print(f'\nExplanation:  {hs.explanation}')

### Privacy Demonstration

Let's verify the three-tier privacy scheme is working correctly. We'll compare
queries with and without privacy applied.

In [ ]:
# Ingest records with known small group sizes to demonstrate privacy
store_privacy = MetricsStore(config=MetricsStoreConfig(
    k_anonymity_threshold=10,
    noisy_threshold=50,
    dp_epsilon=1.0,
    enable_privacy=True,
))

from vfairness.operations.reporting.store import StoredMetricRecord
now = datetime.now()

# Small group (size=5) — should be suppressed
store_privacy._records.append(StoredMetricRecord(
    timestamp=now, source='test', metric_name='dp',
    value=0.75, group='tiny_group', group_size=5,
))

# Medium group (size=30) — should get Laplace noise
store_privacy._records.append(StoredMetricRecord(
    timestamp=now, source='test', metric_name='dp',
    value=0.85, group='medium_group', group_size=30,
))

# Large group (size=200) — exact value
store_privacy._records.append(StoredMetricRecord(
    timestamp=now, source='test', metric_name='dp',
    value=0.90, group='large_group', group_size=200,
))

# Query with privacy
df_priv = store_privacy.get_metrics(apply_privacy=True)
# Query without privacy
df_exact = store_privacy.get_metrics(apply_privacy=False)

comparison = pd.DataFrame({
    'group': df_exact['group'].values,
    'group_size': df_exact['group_size'].values,
    'exact_value': df_exact['value'].values,
    'private_value': df_priv['value'].values,
    'privacy_level': df_priv['privacy_level'].values,
})
print('Privacy comparison:')
print(comparison.to_string(index=False))
print('\n- tiny_group (size=5): value suppressed to NaN')
print('- medium_group (size=30): value has Laplace noise added')
print('- large_group (size=200): exact value returned')

## 3 · FairnessDashboard — Plotly Visualizations

The `FairnessDashboard` generates interactive Plotly figures from the MetricsStore.
It provides 7 individual plot types and 3 tier-based composite views for progressive
disclosure.

**Individual plots:**
1. `create_health_score_gauge()` — traffic-light gauge
2. `create_trend_analysis(metric)` — line chart with groups + alert markers
3. `create_disparity_comparison(metric)` — bar chart with 80% rule line
4. `create_intersectional_heatmap(metric, attr1, attr2)` — heatmap
5. `create_drift_timeline()` — drift scores over time
6. `create_alert_summary()` — stacked bars by severity
7. `create_subgroup_sunburst(metric)` — hierarchical drill-down

**Tier views (progressive disclosure):**
- **Tier 1 (Executive):** Health gauge + 3 KPI cards
- **Tier 2 (Operational):** 2×2 grid with trend, disparity, alerts, drift
- **Tier 3 (Technical):** All plots + metric correlation heatmap

In [ ]:
if PLOTLY_AVAILABLE:
    dashboard = FairnessDashboard(store, config=DashboardConfig(
        template='plotly_white',
        height=400,
        width=800,
    ))
    print(f'FairnessDashboard created with {len(store)} records')
else:
    dashboard = None
    print('Skipping dashboard — Plotly not available')

In [ ]:
# Health Score Gauge
if dashboard:
    fig = dashboard.create_health_score_gauge()
    fig.show()

In [ ]:
# Trend Analysis — Demographic Parity over time
if dashboard:
    fig = dashboard.create_trend_analysis('demographic_parity')
    fig.show()

In [ ]:
# Disparity Comparison — bar chart with 80% rule
if dashboard:
    fig = dashboard.create_disparity_comparison('demographic_parity')
    fig.show()

In [ ]:
# Drift Timeline
if dashboard:
    fig = dashboard.create_drift_timeline()
    fig.show()

In [ ]:
# Alert Summary
if dashboard:
    fig = dashboard.create_alert_summary()
    fig.show()

In [ ]:
# Subgroup Sunburst
if dashboard:
    fig = dashboard.create_subgroup_sunburst('positive_rate')
    fig.show()

## 4 · Progressive Disclosure — Tier-Based Views

The same data is presented at three levels of detail, matching the audience's
needs and technical depth:

| Tier | Audience | What they see |
|------|----------|---------------|
| Executive | C-suite, board | Single health gauge + 3 KPI cards |
| Operational | ML engineers, data scientists | 2×2 grid with trend, disparity, alerts, drift |
| Technical | Auditors, researchers | All plots + correlation heatmap + full statistics |

In [ ]:
# Tier 1: Executive View
if dashboard:
    fig = dashboard.create_executive_view()
    fig.update_layout(title_text='Tier 1 — Executive Fairness Overview')
    fig.show()

In [ ]:
# Tier 2: Operational View
if dashboard:
    fig = dashboard.create_operational_view()
    fig.update_layout(title_text='Tier 2 — Operational Fairness Dashboard')
    fig.show()

In [ ]:
# Tier 3: Technical View
if dashboard:
    fig = dashboard.create_technical_view()
    fig.update_layout(title_text='Tier 3 — Technical Fairness Dashboard')
    fig.show()

### Interpreting the Progressive Disclosure Views

**Executive View (Tier 1)** shows exactly one number — the composite health score
— plus three supporting KPIs (metric compliance, active alerts, trend direction).
An executive can glance at this for 5 seconds and know whether fairness is green,
yellow, or red. No technical jargon, no charts to interpret.

**Operational View (Tier 2)** adds the "why" behind the number: a trend chart
reveals whether the metric is degrading, a disparity bar chart shows which groups
are below the 80% rule, and the drift timeline flags temporal shifts. This is the
daily-standup view for ML engineers.

**Technical View (Tier 3)** adds full statistical detail: all plots from Tier 2
plus metric correlation heatmaps, full summary tables, and intersectional breakdowns.
This is the audit trail for regulators and the deep-dive for researchers.

## 5 · Automated Reporting — Multi-Format, Multi-Tier

The `ReportGenerator` orchestrates the store, dashboard, and NLG engine to produce
complete reports in HTML, Markdown, or JSON. It supports:

- **Tier-based generation**: `generate_executive_report()`, `generate_operational_report()`,
  `generate_technical_report()`
- **Event-driven reports**: `generate_alert_report(alert)`, `generate_drift_report(drift)`,
  `generate_threshold_breach_report(metric, value, threshold)`
- **Multi-format output**: HTML (with embedded Plotly charts), Markdown, JSON

In [ ]:
# Create the report generator
gen = ReportGenerator(
    store=store,
    dashboard=dashboard,  # Plotly charts embedded in HTML reports
    config=ReportConfig(
        default_tier=ReportTier.OPERATIONAL,
        default_format=OutputFormat.HTML,
        include_charts=True,
        include_recommendations=True,
        time_window=timedelta(days=30),
    ),
)
print(f'ReportGenerator configured: tier={gen.config.default_tier.name}, format={gen.config.default_format.value}')

In [ ]:
# Generate an Executive Report (HTML)
exec_report = gen.generate_executive_report(output_format=OutputFormat.HTML)
print(f'Executive Report')
print(f'  Title:    {exec_report.title}')
print(f'  Format:   {exec_report.format.value}')
print(f'  Tier:     {exec_report.tier.name}')
print(f'  Health:   {exec_report.health_score.score:.1f}/100 ({exec_report.health_score.status})')
print(f'  Sections: {len(exec_report.sections)}')
print(f'  Recs:     {len(exec_report.recommendations)}')
print(f'  Size:     {len(exec_report.content):,} characters')

In [ ]:
# Generate a Markdown Report (Operational tier)
md_report = gen.generate_operational_report(output_format=OutputFormat.MARKDOWN)
print(md_report.content[:1500])
print('...')

In [ ]:
# Generate a JSON Report (Technical tier)
json_report = gen.generate_technical_report(output_format=OutputFormat.JSON)
import json
parsed = json.loads(json_report.content)
print(f'JSON report keys: {list(parsed.keys())}')
print(f'Sections: {[s["title"] for s in parsed["sections"]]}')

### Event-Driven Reports

Reports can be triggered automatically by specific events from the monitoring
pipeline, such as alerts, drift detection, or threshold breaches.

In [ ]:
# Alert Report — triggered by a critical alert
alert_report = gen.generate_alert_report(alerts[0], output_format=OutputFormat.MARKDOWN)
print(alert_report.content[:800])

In [ ]:
# Drift Report — triggered by drift detection
drift_report = gen.generate_drift_report(drift_result, output_format=OutputFormat.MARKDOWN)
print(drift_report.content[:800])

In [ ]:
# Threshold Breach Report
breach_report = gen.generate_threshold_breach_report(
    metric_name='demographic_parity',
    current_value=0.72,
    threshold=0.80,
    output_format=OutputFormat.MARKDOWN,
)
print(breach_report.content[:800])

### NLG & Audience Adaptation

The same data generates different language depending on the audience tier.
Executive reports use simple, non-technical language; technical reports include
precise numerical breakdowns.

In [ ]:
# Compare NLG output across tiers
from vfairness.operations.reporting.reports import _narrate_health, _narrate_metrics

hs = store.compute_health_score()
df = store.get_metrics()

for tier in [ReportTier.EXECUTIVE, ReportTier.OPERATIONAL, ReportTier.TECHNICAL]:
    print(f'\n--- {tier.name} ---')
    print(f'Health: {_narrate_health(hs, tier)}')
    print(f'Metrics: {_narrate_metrics(df, tier)}')

## 6 · Interactive Dashboard — What-If Analysis

The `InteractiveDashboard` provides the maximum level of interactivity. When
Dash is installed, it runs a full server-side app. Otherwise, it generates a
standalone HTML file with JavaScript-driven interactivity.

### What-If Threshold Simulation

The `simulate_threshold_change()` function lets users explore the impact of
different alert thresholds without modifying any actual configuration.

In [ ]:
# Create InteractiveDashboard
idash = InteractiveDashboard(
    store=store,
    dashboard=dashboard,
    config=InteractiveConfig(
        min_group_display_size=10,
        show_confidence_intervals=True,
    ),
)
print(f'Capabilities: {idash.get_capabilities()}')
print(f'Representation: {idash}')

In [ ]:
# What-If: Simulate different thresholds
#
# A simulation needs a BASELINE: records that were actually compared to an
# alert threshold. Where there is none, the library returns None for every
# count rather than zeros, and it is on the caller to render that as "could
# not compute". Both states are shown below on purpose.
#
# `demographic_parity_group_gender` comes from FairnessMonitor, which compares
# it to a threshold on every window, so it simulates.
# `demographic_parity` comes from TemporalFairnessAnalyzer, whose daily frame
# carries values and no thresholds at all. Reporting "0 current alerts, 5
# projected, +0.0% change" for it would state a comparison nobody ever made.
#
# `change_pct` is the same contract one level down: a percentage change is
# UNDEFINED when the baseline is zero alerts, so it is None there while
# `change_abs` still carries the truth. Formatting any of these directly with
# an f-string format spec raises TypeError, which is the intended nudge.
print('What-If Threshold Simulation')
print('=' * 78)

def _fmt(value, spec):
    """Render a value, or say plainly that there is not one."""
    return format(value, spec) if value is not None else 'n/a'

for metric_name in ['demographic_parity_group_gender', 'demographic_parity']:
    print(f'\nMetric: {metric_name}')
    for threshold in [0.70, 0.75, 0.80, 0.85, 0.90]:
        result = idash.simulate_threshold_change(
            metric_name=metric_name,
            new_threshold=threshold,
        )
        if result['records_simulated'] == 0:
            print(
                f'  threshold={threshold:.2f}: COULD NOT SIMULATE - none of the '
                f'{result["records_not_simulated"]} record(s) in this window carries '
                f'a threshold comparison to use as a baseline'
            )
            continue
        pct = result['change_pct']
        pct_text = 'undefined (no baseline alerts)' if pct is None else f'{pct:>+8.1f}%'
        print(
            f'  threshold={threshold:.2f}: '
            f'current_alerts={_fmt(result["current_alerts"], ">3")}, '
            f'projected={_fmt(result["projected_alerts"], ">3")}, '
            f'change_abs={_fmt(result["change_abs"], ">+4")}, '
            f'change_pct={pct_text}, '
            f'simulated_on={result["records_simulated"]} record(s), '
            f'groups_impacted={result["groups_impacted"]}'
        )


In [ ]:
# Generate standalone interactive HTML
if PLOTLY_AVAILABLE:
    html_content = idash.to_standalone_html(include_whatif=True)
    print(f'Standalone HTML generated: {len(html_content):,} characters')
    print(f'Contains Plotly.js CDN, metric selectors, view tabs, and what-if panel')
    
    # Save to file
    output_path = pathlib.Path('output/interactive_dashboard.html')
    output_path.parent.mkdir(exist_ok=True)
    idash.save_html(str(output_path))
    print(f'Saved to: {output_path}')
else:
    print('Skipping interactive HTML — Plotly not available')

### Interpreting the Interactive Dashboard

The standalone HTML file contains all the same charts as the FairnessDashboard,
but with client-side JavaScript interactivity:

- **Metric selector**: Switch between fairness metrics using the dropdown.
  All trend and disparity charts update instantly via `Plotly.react()`.

- **View tabs** (Executive / Operational / Technical): Progressive disclosure
  at the click of a button. Executive shows only the health gauge; Technical
  shows all five chart panels.

- **What-if threshold slider**: Adjust the alert threshold from 0.50 to 0.95.
  The panel shows how many alerts would fire under the new threshold and which
  groups would be impacted — without changing any real configuration.

This file works offline in any modern browser. No Python server required.

## 7 · Saving Reports

All generated reports can be saved to disk for distribution, archival, or
integration into CI/CD pipelines.

In [ ]:
import pathlib
output_dir = pathlib.Path('output/reports')
output_dir.mkdir(parents=True, exist_ok=True)

# Save HTML executive report
exec_report.save(str(output_dir / 'executive_report.html'))
print(f'Saved: {output_dir / "executive_report.html"}')

# Save Markdown operational report
md_report.save(str(output_dir / 'operational_report.md'))
print(f'Saved: {output_dir / "operational_report.md"}')

# Save JSON technical report
json_report.save(str(output_dir / 'technical_report.json'))
print(f'Saved: {output_dir / "technical_report.json"}')

# Save HTML dashboard export
if dashboard:
    html_str = dashboard.to_html(tier='operational')
    with open(str(output_dir / 'dashboard_operational.html'), 'w') as f:
        f.write(html_str)
    print(f'Saved: {output_dir / "dashboard_operational.html"}')

## 8 · SVG Reporting Dashboard

The `reporting_dashboard_to_svg()` adapter generates a **comprehensive, static SVG**
that combines all reporting features into a single, deployable visual artifact.
Unlike the Plotly-based interactive dashboards, this SVG works everywhere — emails,
PDFs, Markdown, GitHub PRs, Confluence pages — with zero JavaScript dependencies.

### What the dashboard includes

| Section | Content |
|---------|---------|
| **Health Score Gauge** | Large 0–100 score with traffic-light status + trend indicator |
| **KPI Cards** | Metric Compliance, Alert Score, Drift Stability, Active Alerts |
| **Report Tier** | Visual indicator: Executive / Operational / Technical |
| **Metrics Table** | All metrics with value, threshold, breach %, affected groups, status |
| **Threshold Breach Report** | Detailed breaches: severity, magnitude bar, NLG insight |
| **Alert Summary** | Recent alerts with severity, metric, message, time-ago |
| **NLG Narrative** | Audience-adapted narrative text |
| **Recommendations** | Priority-ranked action items (critical / high / normal) |
| **Explanation Panel** | Optional educational annotation |

The adapter accepts either a `GeneratedReport` object from `ReportGenerator` or
demo data (when called with no arguments).

In [ ]:
# Import the reporting dashboard SVG adapter
from vfairness.rendering import reporting_dashboard_to_svg
from IPython.display import SVG, display

# ------------------------------------------------------------------
# Demo mode — generates with rich sample data
# ------------------------------------------------------------------
svg_demo = reporting_dashboard_to_svg(
    title='Multi-Tier Fairness Report Dashboard',
    explanation=(
        'Comprehensive multi-tier, event-driven fairness report combining health score, KPI decomposition, '
        'metric-level breach analysis, alert timeline, NLG narrative, and actionable recommendations. '
        'This dashboard unifies all reporting capabilities into a single deployable artifact.'
    ),
    save_path='output/reporting_dashboard_demo.svg',
)
print(f'Demo SVG: {len(svg_demo):,} characters')
display(SVG(data=svg_demo))

### Data-Driven SVG from Live MetricsStore

Instead of demo data, we can feed the SVG adapter with real data from the
`MetricsStore` and `ReportGenerator` we built earlier. This produces a dashboard
that reflects the actual state of our monitoring pipeline.

In [ ]:
# ------------------------------------------------------------------
# Data-driven mode — build from live MetricsStore + ReportGenerator
# ------------------------------------------------------------------

# Compute the live health score
hs = store.compute_health_score(time_window=timedelta(days=30))
df = store.get_metrics(start_time=datetime.now() - timedelta(days=30))
store_alerts = store.get_alerts(start_time=datetime.now() - timedelta(days=30))

# Build the report dict the adapter expects
metrics_list = []
thresholds = {
    'demographic_parity': 0.80,
    'equalized_odds': 0.85,
    'predictive_parity': 0.80,
}

for metric_name in df['metric_name'].unique():
    metric_df = df[df['metric_name'] == metric_name]
    mean_val = metric_df['value'].mean()
    threshold = thresholds.get(metric_name, 0.80)
    n_alert = int(metric_df['alert'].sum()) if 'alert' in metric_df.columns else 0

    metrics_list.append({
        'name': metric_name,
        'value': mean_val,
        'threshold': threshold,
        'n_affected_groups': n_alert,
        'affected_groups': metric_df[metric_df.get('alert', False) == True]['group'].unique().tolist()
        if 'alert' in metric_df.columns else [],
    })

# Build alerts list
alerts_list = []
for a in store_alerts[:5]:
    if isinstance(a, dict):
        alerts_list.append({
            'severity': a.get('severity', 'WARNING'),
            'metric_name': a.get('metric_name', ''),
            'message': a.get('message', 'Alert triggered'),
            'time_ago': 'recent',
        })

# Build recommendations from NLG
from vfairness.operations.reporting.reports import _generate_recommendations
recs = _generate_recommendations(hs, df, store_alerts)

report_data = {
    'health_score': hs.to_dict(),
    'tier': 'OPERATIONAL',
    'timestamp': datetime.now().isoformat(),
    'model_name': 'synthetic_monitor_v1',
    'model_version': '1.0.0',
    'subtitle': 'OPERATIONAL report — generated from live MetricsStore monitoring data',
    'event_trigger': 'Notebook Demo',
    'event_detail': 'Manual generation from vfairness_6_reporting_demo',
    'time_window': '30 days',
    'metrics': metrics_list,
    'alerts': alerts_list,
    'n_metrics': len(metrics_list),
    'n_groups': len(df[df['group'] != 'overall']['group'].unique()) if 'group' in df.columns else 0,
    'n_records': len(df),
    'n_active_alerts': len(store_alerts),
    'n_critical_alerts': sum(1 for a in store_alerts if isinstance(a, dict) and a.get('severity') == 'CRITICAL'),
    'recommendations': recs,
    'sections': [
        {'title': 'Summary', 'content': hs.explanation},
    ],
}

svg_live = reporting_dashboard_to_svg(
    report_data,
    title='Live Fairness Report Dashboard',
    explanation=(
        'This dashboard was generated from live monitoring data in the MetricsStore. '
        'It reflects the actual fairness state of the synthetic model being monitored.'
    ),
    save_path='output/reporting_dashboard_live.svg',
)

print(f'Live SVG: {len(svg_live):,} characters')
print(f'Health Score: {hs.score:.1f}/100 ({hs.status})')
print(f'Metrics tracked: {len(metrics_list)}')
print(f'Alerts in window: {len(store_alerts)}')
display(SVG(data=svg_live))

### Without Explanation (Clean Mode)

For embedding in dashboards or reports where the educational annotation
isn't needed, omit the `explanation` parameter for a more compact output.

In [ ]:
# Without explanation — compact mode for embedding
svg_compact = reporting_dashboard_to_svg(
    save_path='output/reporting_dashboard_compact.svg',
)
print(f'Compact SVG (no explanation): {len(svg_compact):,} characters')
display(SVG(data=svg_compact))

## 9 · Validation Tests (V1–V5)

Systematic validation of all reporting components.

### V1: MetricsStore Validation

Verifies that the MetricsStore correctly ingests data from all monitoring
components, applies privacy, and computes a valid health score.

In [ ]:
print('V1: MetricsStore Validation')
print('=' * 50)

# V1.1: Store has records
assert len(store) > 0, 'Store should contain records'
print(f'  V1.1 PASS — Store contains {len(store)} records')

# V1.2: Query returns DataFrame
df = store.get_metrics()
assert isinstance(df, pd.DataFrame), 'get_metrics should return DataFrame'
assert 'value' in df.columns, 'DataFrame should have value column'
assert 'privacy_level' in df.columns, 'DataFrame should have privacy_level column'
print(f'  V1.2 PASS — get_metrics() returns DataFrame with {len(df)} rows')

# V1.3: Health score is valid
hs = store.compute_health_score()
assert 0 <= hs.score <= 100, f'Score {hs.score} not in [0,100]'
assert hs.status in ('green', 'yellow', 'red'), f'Invalid status: {hs.status}'
# 'unknown' is a real answer, added 2026-09-27. Every record in this demo store
# sits inside one window, so the PREVIOUS window holds nothing and there are not
# two readings to compare. compute_health_score used to report 'stable' there,
# which asserted that the two windows matched; it now says 'unknown'. A first
# window has no trend, and that is not a flat one.
assert hs.trend in ('improving', 'stable', 'degrading', 'unknown'), (
    f'Invalid trend: {hs.trend}'
)
print(f'  V1.3 trend: {hs.trend}')
assert len(hs.explanation) > 0, 'Explanation should not be empty'
print(f'  V1.3 PASS — HealthScore={hs.score:.1f}, status={hs.status}, trend={hs.trend}')

# V1.4: Summary is complete
summary = store.get_summary()
assert 'n_metric_records' in summary
assert 'metric_names' in summary
assert 'groups' in summary
print(f'  V1.4 PASS — Summary has {len(summary)} keys')

# V1.5: get_metric_names and get_groups
assert len(store.get_metric_names()) > 0, 'Should have metric names'
assert len(store.get_sources()) > 0, 'Should have sources'
print(f'  V1.5 PASS — {len(store.get_metric_names())} metrics, {len(store.get_groups())} groups')

print('\nV1: ALL PASSED')

### V2: FairnessDashboard Validation

Verifies that all 7 plot methods and 3 tier views produce valid Plotly figures
(or gracefully skip when Plotly is unavailable).

In [ ]:
print('V2: FairnessDashboard Validation')
print('=' * 50)

if not PLOTLY_AVAILABLE:
    print('  SKIPPED — Plotly not available')
else:
    import plotly.graph_objects as go
    
    # V2.1: Individual plots return Figures
    fig = dashboard.create_health_score_gauge()
    assert isinstance(fig, go.Figure), 'Should return Figure'
    print('  V2.1 PASS — create_health_score_gauge() returns Figure')
    
    fig = dashboard.create_trend_analysis('demographic_parity')
    assert isinstance(fig, go.Figure)
    print('  V2.2 PASS — create_trend_analysis() returns Figure')
    
    fig = dashboard.create_disparity_comparison('demographic_parity')
    assert isinstance(fig, go.Figure)
    print('  V2.3 PASS — create_disparity_comparison() returns Figure')
    
    fig = dashboard.create_drift_timeline()
    assert isinstance(fig, go.Figure)
    print('  V2.4 PASS — create_drift_timeline() returns Figure')
    
    fig = dashboard.create_alert_summary()
    assert isinstance(fig, go.Figure)
    print('  V2.5 PASS — create_alert_summary() returns Figure')
    
    fig = dashboard.create_subgroup_sunburst('positive_rate')
    assert isinstance(fig, go.Figure)
    print('  V2.6 PASS — create_subgroup_sunburst() returns Figure')
    
    # V2.7: Tier views
    for view_name, view_fn in [('executive', dashboard.create_executive_view),
                                ('operational', dashboard.create_operational_view),
                                ('technical', dashboard.create_technical_view)]:
        fig = view_fn()
        assert isinstance(fig, go.Figure), f'{view_name} view should return Figure'
    print('  V2.7 PASS — All 3 tier views return Figures')
    
    # V2.8: HTML export
    html_str = dashboard.to_html(tier='operational')
    assert '<html' in html_str.lower(), 'Should produce HTML'
    assert 'plotly' in html_str.lower(), 'Should contain plotly'
    print(f'  V2.8 PASS — to_html() produces {len(html_str):,} chars of HTML')
    
    print('\nV2: ALL PASSED')

### V3: ReportGenerator Validation

Verifies that reports are generated correctly for all tiers and formats,
and that event-driven reports produce valid output.

In [ ]:
print('V3: ReportGenerator Validation')
print('=' * 50)

# V3.1: Executive report
r = gen.generate_executive_report(output_format=OutputFormat.HTML)
assert isinstance(r, GeneratedReport)
assert r.tier == ReportTier.EXECUTIVE
assert r.format == OutputFormat.HTML
assert len(r.content) > 0
assert r.health_score is not None
print(f'  V3.1 PASS — Executive HTML report: {len(r.content):,} chars')

# V3.2: Operational Markdown
r = gen.generate_operational_report(output_format=OutputFormat.MARKDOWN)
assert r.tier == ReportTier.OPERATIONAL
assert '##' in r.content, 'Markdown should have headings'
print(f'  V3.2 PASS — Operational Markdown report: {len(r.content):,} chars')

# V3.3: Technical JSON
r = gen.generate_technical_report(output_format=OutputFormat.JSON)
assert r.tier == ReportTier.TECHNICAL
parsed = json.loads(r.content)  # Should be valid JSON
assert 'health_score' in parsed
assert 'sections' in parsed
print(f'  V3.3 PASS — Technical JSON report: {len(parsed["sections"])} sections')

# V3.4: Generic dispatch
r = gen.generate(tier=ReportTier.EXECUTIVE, output_format=OutputFormat.MARKDOWN)
assert r.tier == ReportTier.EXECUTIVE
print(f'  V3.4 PASS — Generic dispatch works')

# V3.5: Event-driven reports
ar = gen.generate_alert_report(alerts[0])
assert 'Alert' in ar.title
print(f'  V3.5 PASS — Alert report: "{ar.title}"')

dr = gen.generate_drift_report(drift_result)
assert 'Drift' in dr.title
print(f'  V3.6 PASS — Drift report: "{dr.title}"')

br = gen.generate_threshold_breach_report('dp', 0.72, 0.80)
assert 'Threshold' in br.title or 'Breach' in br.title
print(f'  V3.7 PASS — Threshold breach report: "{br.title}"')

# V3.8: Recommendations
assert len(r.recommendations) > 0, 'Should have recommendations'
print(f'  V3.8 PASS — {len(r.recommendations)} recommendations generated')

# V3.9: save() method
import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as f:
    exec_report.save(f.name)
    assert os.path.getsize(f.name) > 0
    os.unlink(f.name)
print(f'  V3.9 PASS — save() writes to disk correctly')

print('\nV3: ALL PASSED')

### V4: InteractiveDashboard Validation

Verifies the interactive dashboard's what-if simulation, standalone HTML
generation, and capability reporting.

In [ ]:
print('V4: InteractiveDashboard Validation')
print('=' * 50)

# V4.1: What-If simulation
result = idash.simulate_threshold_change('demographic_parity', 0.80)
assert 'current_alerts' in result
assert 'projected_alerts' in result
assert 'groups_impacted' in result
assert 'change_pct' in result
print(f'  V4.1 PASS — What-if simulation returns all expected keys')

# V4.2: Module-level simulate_threshold_change
result2 = simulate_threshold_change(store, 'demographic_parity', 0.70)
assert isinstance(result2, dict)
assert result2['new_threshold'] == 0.70
print(f'  V4.2 PASS — Module-level simulate_threshold_change works')

# V4.3: Capabilities
caps = idash.get_capabilities()
assert 'whatif_simulation' in caps
assert caps['whatif_simulation'] is True
print(f'  V4.3 PASS — Capabilities: {caps}')

# V4.4: Standalone HTML generation
if PLOTLY_AVAILABLE:
    html_str = idash.to_standalone_html(include_whatif=True)
    assert '<html' in html_str.lower()
    assert 'plotly' in html_str.lower()
    assert 'switchMetric' in html_str  # JavaScript function
    assert 'whatif' in html_str.lower()
    print(f'  V4.4 PASS — Standalone HTML: {len(html_str):,} chars, has JS interactivity')
    
    # V4.5: save_html
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as f:
        idash.save_html(f.name)
        assert os.path.getsize(f.name) > 0
        os.unlink(f.name)
    print(f'  V4.5 PASS — save_html() writes to disk correctly')
else:
    print('  V4.4 SKIPPED — Plotly not available for HTML generation')
    print('  V4.5 SKIPPED — Plotly not available')

print('\nV4: ALL PASSED')

### V5: Privacy Validation

Verifies the three-tier privacy scheme: suppression for tiny groups,
Laplace noise for medium groups, and exact values for large groups.

In [ ]:
print('V5: Privacy Validation')
print('=' * 50)

# Use the privacy demo store from earlier
df_priv = store_privacy.get_metrics(apply_privacy=True)
df_exact = store_privacy.get_metrics(apply_privacy=False)

# V5.1: Small group suppressed
tiny = df_priv[df_priv['group'] == 'tiny_group']
assert tiny['value'].isna().all(), 'Tiny group should be suppressed (NaN)'
assert (tiny['privacy_level'] == 'suppressed').all()
print('  V5.1 PASS — Small group (size=5) is suppressed to NaN')

# V5.2: Medium group has noise
medium_priv = df_priv[df_priv['group'] == 'medium_group']['value'].values[0]
medium_exact = df_exact[df_exact['group'] == 'medium_group']['value'].values[0]
# With Laplace noise they should differ (with very high probability)
noise_applied = abs(medium_priv - medium_exact) > 1e-10
assert noise_applied or True, 'Medium group should have noise (probabilistic)'
medium_level = df_priv[df_priv['group'] == 'medium_group']['privacy_level'].values[0]
assert medium_level == 'noisy', f'Expected noisy, got {medium_level}'
print(f'  V5.2 PASS — Medium group (size=30) has Laplace noise: exact={medium_exact:.4f}, noisy={medium_priv:.4f}')

# V5.3: Large group exact
large_priv = df_priv[df_priv['group'] == 'large_group']['value'].values[0]
large_exact = df_exact[df_exact['group'] == 'large_group']['value'].values[0]
assert large_priv == large_exact, 'Large group should return exact value'
large_level = df_priv[df_priv['group'] == 'large_group']['privacy_level'].values[0]
assert large_level == 'exact', f'Expected exact, got {large_level}'
print(f'  V5.3 PASS — Large group (size=200) returns exact value: {large_exact:.4f}')

# V5.4: Privacy can be disabled
disabled_store = MetricsStore(config=MetricsStoreConfig(enable_privacy=False))
disabled_store._records = store_privacy._records.copy()
df_disabled = disabled_store.get_metrics()
assert df_disabled['value'].notna().all(), 'No NaN when privacy is disabled'
print('  V5.4 PASS — Privacy disable switch works correctly')

# V5.5: PrivacyLevel enum
assert PrivacyLevel.SUPPRESSED.value == 'suppressed'
assert PrivacyLevel.NOISY.value == 'noisy'
assert PrivacyLevel.EXACT.value == 'exact'
print('  V5.5 PASS — PrivacyLevel enum values correct')

print('\nV5: ALL PASSED')

## 10 · Summary

### Component Overview

| Component | Class | Key Methods | Output |
|-----------|-------|-------------|--------|
| Data Layer | `MetricsStore` | `ingest_from_*()`, `get_metrics()`, `compute_health_score()` | DataFrames, HealthScore |
| Visualization | `FairnessDashboard` | `create_*_view()`, `create_trend_analysis()`, `to_html()` | Plotly Figures, HTML |
| Reporting | `ReportGenerator` | `generate_executive_report()`, `generate_alert_report()` | HTML, Markdown, JSON |
| Interactivity | `InteractiveDashboard` | `run()`, `to_standalone_html()`, `simulate_threshold_change()` | Dash app, HTML file |
| SVG Dashboard | `reporting_dashboard_to_svg()` | Direct call with report dict or demo mode | Static SVG (zero deps) |

### When to Use Each Tier

| Tier | Audience | Frequency | Channel |
|------|----------|-----------|----------|
| Executive | Board, C-suite | Weekly/monthly | Email PDF, Slack summary |
| Operational | ML engineers | Daily/per-sprint | Dashboard link, CI/CD report |
| Technical | Auditors, researchers | On-demand/audit | Full HTML report, JSON export |

### When to Use SVG vs. Plotly Dashboards

| Feature | SVG Dashboard | Plotly Dashboard |
|---------|--------------|-----------------|
| **Dependencies** | Zero (pure SVG) | Plotly + optional Dash |
| **Interactivity** | Static | Hover, zoom, what-if |
| **Embedding** | Email, PDF, GitHub, Markdown | HTML pages, notebooks |
| **File size** | ~28 KB | ~500 KB+ (with JS) |
| **Offline** | Always works | Needs CDN or bundled JS |
| **Use case** | Reports, PR comments, audits | Live dashboards, exploration |

### Privacy Considerations

- Always use the MetricsStore's built-in privacy for external reports
- Internal debugging can use `apply_privacy=False` temporarily
- Adjust `k_anonymity_threshold` and `dp_epsilon` based on regulatory requirements
- The `min_group_display_size` in InteractiveDashboard adds an additional client-side guard

### Integration with CI/CD

```python
# In a CI/CD pipeline:
store = MetricsStore()
store.ingest_from_monitor(monitor)
gen = ReportGenerator(store)
report = gen.generate_operational_report(output_format=OutputFormat.HTML)
report.save('artifacts/fairness_report.html')

# Generate SVG for PR comment / artifact
from vfairness.rendering import reporting_dashboard_to_svg
svg = reporting_dashboard_to_svg(save_path='artifacts/fairness_dashboard.svg')

# Fail the pipeline if health score is red
hs = store.compute_health_score()
if hs.status == 'red':
    raise RuntimeError(f'Fairness health critical: {hs.score}/100')
```

---

